## Dataset Embedding Experiments with clustering and cross encoder

In [58]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [59]:
import pandas as pd

In [60]:
from sentence_transformers import SentenceTransformer, CrossEncoder

bi_encoder = SentenceTransformer('all-mpnet-base-v2')  

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')



In [61]:
jd_df = pd.read_excel("../1_data_cleaning/filtered_jd_sections2.xlsx")


In [62]:
jd_df.head()

,job_description,location_cleaned,job_title,jd_duties,jd_requirements,jd_education
0,Job SummaryDo you have a strong aptitude for w...,"Natick, MA",Content Developer for MATLAB Code Generation,Software components make up an ever larger par...,Job SummaryDo you have a strong aptitude for w...,Minimum Qualifications A bachelor's degree and...
1,Overview External: Chevron is one of the world...,"Houston, TX",Land Assistant,Overview External: Chevron is one of the world...,"Prepare, and secure appropriate approvals for ...",Preferred Qualifications: Bachelor's degree
2,Overview: The Hartman Non-Profit is seeking to...,"Houston, TX",Project Manager,Overview: The Hartman Non-Profit is seeking to...,Required Abilities and Experience: · A strong ...,· A minimum of a Bachelor’s degree in a busine...
3,City: Houston State:Texas Postal/Zip Code: 770...,"Houston, TX",Asphalt Quality Control Manager -Houston,Our operations span the nation from Montana to...,Other Requirements Display a professional and ...,Qualifications Bachelor’s degree (B
4,Hiring! Immediate openings for Customer Suppor...,"San Antonio, TX",Customer Support Specialist!,Immediate openings for Customer Support Specia...,Hiring\nWe want your excellent customer servic...,High School Diploma/GED


In [63]:
# create semantic sentence for each row
def jd_build_semantic_sentence(row):
    desc = str(row.get("job_description", ""))
    duties = str(row.get("jd_duties", ""))
    req = str(row.get("jd_requirements", ""))
    edu = str(row.get("jd_education", ""))
    title = str(row.get("job_title", ""))

    return (
        f"Job Posting:\n"
        f"- Job Title: {title}\n"
        f"- Description: {desc}\n"
        f"- Responsibilities: {duties}\n"
        f"- Requirements: {req}\n"
        f"- Preferred Education: {edu}"
    )



jd_df["semantic_sentence"] = jd_df.apply(jd_build_semantic_sentence, axis=1)
jd_df.head()



,job_description,location_cleaned,job_title,jd_duties,jd_requirements,jd_education,semantic_sentence
0,Job SummaryDo you have a strong aptitude for w...,"Natick, MA",Content Developer for MATLAB Code Generation,Software components make up an ever larger par...,Job SummaryDo you have a strong aptitude for w...,Minimum Qualifications A bachelor's degree and...,Job Posting:\n- Job Title: Content Developer f...
1,Overview External: Chevron is one of the world...,"Houston, TX",Land Assistant,Overview External: Chevron is one of the world...,"Prepare, and secure appropriate approvals for ...",Preferred Qualifications: Bachelor's degree,Job Posting:\n- Job Title: Land Assistant\n- D...
2,Overview: The Hartman Non-Profit is seeking to...,"Houston, TX",Project Manager,Overview: The Hartman Non-Profit is seeking to...,Required Abilities and Experience: · A strong ...,· A minimum of a Bachelor’s degree in a busine...,Job Posting:\n- Job Title: Project Manager\n- ...
3,City: Houston State:Texas Postal/Zip Code: 770...,"Houston, TX",Asphalt Quality Control Manager -Houston,Our operations span the nation from Montana to...,Other Requirements Display a professional and ...,Qualifications Bachelor’s degree (B,Job Posting:\n- Job Title: Asphalt Quality Con...
4,Hiring! Immediate openings for Customer Suppor...,"San Antonio, TX",Customer Support Specialist!,Immediate openings for Customer Support Specia...,Hiring\nWe want your excellent customer servic...,High School Diploma/GED,Job Posting:\n- Job Title: Customer Support Sp...


In [64]:
resume_df = pd.read_csv("../1_data_cleaning/resume_cleaned_100.csv")

In [65]:
resume_df.head(10)

,career_objective,skills,degree_names,major_field_of_studies,positions,responsibilities
0,Experienced product development Engineer and m...,"['Microsoft Office', 'Microsoft Project', 'Pro...",['Bachelor of Science'],['Mechanical Engineering'],"['Engineering Manager', 'Project Engineer II',...",Management Trainee\nMechanical Systems\nMainte...
1,"I am a software engineer, and I want to work o...","['C++', 'Python', 'Firebase', 'Flutter', 'Tens...",['B.Tech'],['Computers'],['SDE'],Recruitment Coordination\nAppointment Manageme...
2,I desire to work for a company that provides c...,"['Word', 'SAP Time Approval', 'Excel', 'Travel...",['Associate of Arts'],['Administrative Assistance'],"['ENGINEERING COORDINATOR', 'FACILITIES TEMP',...",Machine Learning Design\nData Analysis\nModel ...
3,As a Data Analyst I always look into more inno...,"['Machine Learning', 'Artificial Intelligence'...","['B.Tech', 'M.Tech']","[None, None]",['Data Analyst'],Mikrotik Router Configuration\nOLT Device Setu...
4,Financial and Accounting professional with exp...,['Power User of Microsoft Excel Epicor NetSuit...,['Bachelor of Business Administration'],['Accounting'],"['Senior Accountant', 'Senior Accountant/Finan...",Design Creation\nCAD Drawings\nDesign Optimiza...
5,"Fresher starting out with Business Analysis, a...","['Business Analyst', 'Data Analysis', 'Busines...",['BBA'],['N/A'],['Part-Time Analyst'],"Full Stack Development\nFront-end: ReactJS, Ne..."
6,Data Scientist working on problems related to ...,"['Java', 'C++', 'Python', 'Machine Learning', ...","['B.Tech', 'M.Tech']","['Computer Science', 'Computer Science Enginee...",['Data Scientist'],Machine Learning Design\nData Analysis\nModel ...
7,Financial Accountant specializing in financial...,"['account reconciliations', 'accounting', 'acc...","['MBA', 'B.Com', 'Diploma']","['Finance and IT', 'Mgt Hons', 'Computer Appli...","['Accountant', 'Consultant', 'N/A']",Machinery Maintenance\nTroubleshooting\nReport...
8,I am fresher data analyst starting out in ERP ...,"['MIS reporting', 'Advanced Excel', 'Dashboard...","['B.A (Economics)', 'M.A (Economics), Correspo...","['Economics', 'Economics']",['MIS Analyst'],Management Trainee\nMechanical Systems\nMainte...
9,Certified Data analyst with a degree in Electr...,"['Python', 'Machine Learning', 'MySQL', 'Data ...",['B.Tech/B.E.'],['Electronics/Telecommunication'],['Associate Analyst'],Project Design\nData Analysis\nACCORD/Alliance...


In [66]:
# create semantic sentences for resumes
def resume_build_semantic_sentence(row):
    def clean(value):
        if value is None:
            return ""
        if isinstance(value, list):
            return ", ".join([str(v) for v in value])
        return str(value).strip()

    career = clean(row.get("career_objective", ""))
    skills = clean(row.get("skills", ""))
    degrees = clean(row.get("degree_names", ""))
    majors = clean(row.get("major_field_of_studies", ""))
    work_exp = clean(row.get("responsibilities", ""))

    return (
        "Candidate Profile:\n"
        f"- Career Objective: {career}\n"
        f"- Work Experience: {work_exp}\n"
        f"- Skills: {skills}\n"
        f"- Degrees: {degrees}\n"
        f"- Major Field(s) of Study: {majors}"
    )


In [67]:
# create semantic sentences for resumes
resume_df["semantic_sentence"] = resume_df.apply(resume_build_semantic_sentence, axis=1)
resume_df.head()


,career_objective,skills,degree_names,major_field_of_studies,positions,responsibilities,semantic_sentence
0,Experienced product development Engineer and m...,"['Microsoft Office', 'Microsoft Project', 'Pro...",['Bachelor of Science'],['Mechanical Engineering'],"['Engineering Manager', 'Project Engineer II',...",Management Trainee\nMechanical Systems\nMainte...,Candidate Profile:\n- Career Objective: Experi...
1,"I am a software engineer, and I want to work o...","['C++', 'Python', 'Firebase', 'Flutter', 'Tens...",['B.Tech'],['Computers'],['SDE'],Recruitment Coordination\nAppointment Manageme...,Candidate Profile:\n- Career Objective: I am a...
2,I desire to work for a company that provides c...,"['Word', 'SAP Time Approval', 'Excel', 'Travel...",['Associate of Arts'],['Administrative Assistance'],"['ENGINEERING COORDINATOR', 'FACILITIES TEMP',...",Machine Learning Design\nData Analysis\nModel ...,Candidate Profile:\n- Career Objective: I desi...
3,As a Data Analyst I always look into more inno...,"['Machine Learning', 'Artificial Intelligence'...","['B.Tech', 'M.Tech']","[None, None]",['Data Analyst'],Mikrotik Router Configuration\nOLT Device Setu...,Candidate Profile:\n- Career Objective: As a D...
4,Financial and Accounting professional with exp...,['Power User of Microsoft Excel Epicor NetSuit...,['Bachelor of Business Administration'],['Accounting'],"['Senior Accountant', 'Senior Accountant/Finan...",Design Creation\nCAD Drawings\nDesign Optimiza...,Candidate Profile:\n- Career Objective: Financ...


In [68]:
# add index columns for both datasets
jd_df["jd_index"] = jd_df.index
resume_df["resume_index"] = resume_df.index


In [69]:
def embed_sentence(sentence):
    return bi_encoder.encode(sentence)


In [70]:
# embed JD semantic sentences
jd_df["semantic_emb"] = jd_df["semantic_sentence"].apply(embed_sentence)
jd_df.head()

,job_description,location_cleaned,job_title,jd_duties,jd_requirements,jd_education,semantic_sentence,jd_index,semantic_emb
0,Job SummaryDo you have a strong aptitude for w...,"Natick, MA",Content Developer for MATLAB Code Generation,Software components make up an ever larger par...,Job SummaryDo you have a strong aptitude for w...,Minimum Qualifications A bachelor's degree and...,Job Posting:\n- Job Title: Content Developer f...,0,"[-0.010107441, -0.023181697, -0.041947138, -0...."
1,Overview External: Chevron is one of the world...,"Houston, TX",Land Assistant,Overview External: Chevron is one of the world...,"Prepare, and secure appropriate approvals for ...",Preferred Qualifications: Bachelor's degree,Job Posting:\n- Job Title: Land Assistant\n- D...,1,"[0.040389102, 0.07183061, -0.0014597354, -0.04..."
2,Overview: The Hartman Non-Profit is seeking to...,"Houston, TX",Project Manager,Overview: The Hartman Non-Profit is seeking to...,Required Abilities and Experience: · A strong ...,· A minimum of a Bachelor’s degree in a busine...,Job Posting:\n- Job Title: Project Manager\n- ...,2,"[0.036204915, 0.08343988, -0.005588149, 0.0499..."
3,City: Houston State:Texas Postal/Zip Code: 770...,"Houston, TX",Asphalt Quality Control Manager -Houston,Our operations span the nation from Montana to...,Other Requirements Display a professional and ...,Qualifications Bachelor’s degree (B,Job Posting:\n- Job Title: Asphalt Quality Con...,3,"[-0.015358253, 0.04020662, -0.009223265, 0.023..."
4,Hiring! Immediate openings for Customer Suppor...,"San Antonio, TX",Customer Support Specialist!,Immediate openings for Customer Support Specia...,Hiring\nWe want your excellent customer servic...,High School Diploma/GED,Job Posting:\n- Job Title: Customer Support Sp...,4,"[0.018529275, -0.050914153, -0.04652271, -0.04..."


In [71]:
# embed resume semantic sentences
resume_df["semantic_emb"] = resume_df["semantic_sentence"].apply(embed_sentence)
resume_df.head()

,career_objective,skills,degree_names,major_field_of_studies,positions,responsibilities,semantic_sentence,resume_index,semantic_emb
0,Experienced product development Engineer and m...,"['Microsoft Office', 'Microsoft Project', 'Pro...",['Bachelor of Science'],['Mechanical Engineering'],"['Engineering Manager', 'Project Engineer II',...",Management Trainee\nMechanical Systems\nMainte...,Candidate Profile:\n- Career Objective: Experi...,0,"[0.012360605, -0.015497627, -0.0399319, -0.043..."
1,"I am a software engineer, and I want to work o...","['C++', 'Python', 'Firebase', 'Flutter', 'Tens...",['B.Tech'],['Computers'],['SDE'],Recruitment Coordination\nAppointment Manageme...,Candidate Profile:\n- Career Objective: I am a...,1,"[0.036700238, 0.025367294, -0.030690191, -0.05..."
2,I desire to work for a company that provides c...,"['Word', 'SAP Time Approval', 'Excel', 'Travel...",['Associate of Arts'],['Administrative Assistance'],"['ENGINEERING COORDINATOR', 'FACILITIES TEMP',...",Machine Learning Design\nData Analysis\nModel ...,Candidate Profile:\n- Career Objective: I desi...,2,"[0.031198328, 0.022536723, -0.027450446, -0.05..."
3,As a Data Analyst I always look into more inno...,"['Machine Learning', 'Artificial Intelligence'...","['B.Tech', 'M.Tech']","[None, None]",['Data Analyst'],Mikrotik Router Configuration\nOLT Device Setu...,Candidate Profile:\n- Career Objective: As a D...,3,"[0.0232989, 0.053408287, -0.053814877, -0.0603..."
4,Financial and Accounting professional with exp...,['Power User of Microsoft Excel Epicor NetSuit...,['Bachelor of Business Administration'],['Accounting'],"['Senior Accountant', 'Senior Accountant/Finan...",Design Creation\nCAD Drawings\nDesign Optimiza...,Candidate Profile:\n- Career Objective: Financ...,4,"[0.017975513, 0.055254363, -0.023993967, -0.05..."


In [73]:
# convert to numpy array
import numpy as np

resume_df["semantic_emb"] = resume_df["semantic_emb"].apply(lambda x: np.array(x))
jd_df["semantic_emb"] = jd_df["semantic_emb"].apply(lambda x: np.array(x))


In [ ]:
# calculate  the similarity score
from sklearn.metrics.pairwise import cosine_similarity

def get_coarse_candidates(resume_embedding, jd_df, top_k=50):
    candidates = jd_df.copy()

    candidates = candidates[
        candidates["semantic_emb"].apply(lambda x: isinstance(x, (list, np.ndarray)) and len(x) > 0)
    ]

    sims = cosine_similarity([resume_embedding], list(candidates["semantic_emb"]))[0]

    sims_percent = (sims + 1) / 2 * 100
    candidates["sim"] = sims_percent

    return candidates.sort_values("sim", ascending=False).head(top_k)




In [ ]:
def rerank_with_cross_encoder(resume_sentence, candidate_df, cross_encoder, top_k=3):
    if candidate_df is None or len(candidate_df) == 0:
        return []

    pairs = [
        (resume_sentence, jd_sent)
        for jd_sent in candidate_df["semantic_sentence"]
    ]

    scores = cross_encoder.predict(pairs)

    # Min–Max normalization
    min_s, max_s = scores.min(), scores.max()
    normalized = (scores - min_s) / (max_s - min_s + 1e-8)
    
    candidate_df = candidate_df.copy()
    candidate_df["ce_score"] = (normalized * 100).round(2)

    top_matches = candidate_df.sort_values("ce_score", ascending=False).head(top_k)

    return top_matches[["jd_index", "job_title", "location_cleaned", "ce_score"]]



In [ ]:
def match_resume(resume_row, jd_df, k_coarse=50, k_final=3):
    resume_emb = np.array(resume_row["semantic_emb"])
    resume_sentence = resume_row["semantic_sentence"]

    # coarse retrieve
    candidates = get_coarse_candidates(
        resume_embedding=resume_emb,
        jd_df=jd_df,
        top_k=k_coarse
    )

    # cross encoder rerank
    final_matches = rerank_with_cross_encoder(
        resume_sentence=resume_sentence,
        candidate_df=candidates,
        cross_encoder=cross_encoder,
        top_k=k_final
    )

    return final_matches


In [86]:
for i in range(5):
    row = resume_df.iloc[i]
    top_matches = match_resume(row, jd_df)

    print(f"================ Resume #{i} ================")
    print(row["responsibilities"][:400], "...\n")
    print(top_matches)
    print("-----------------------------------------------------\n")


================ Resume #0 ================
Management Trainee
Mechanical Systems
Maintenance & Troubleshooting
Performance Analysis
Project Support
Process Improvement
Training & Development
Administrative Support ...

     jd_index        job_title location_cleaned    ce_score
213       213  Product Manager       Durham, NC  100.000000
887       887  Product Manager       Boston, MA   92.769997
878       878  Product Analyst       Boston, MA   80.190002
-----------------------------------------------------

================ Resume #1 ================
Recruitment Coordination
Appointment Management
Selection Criteria
Employee Orientation
Performance Evaluation
HR Database Management
Report Compilation
Documentation
Event Coordination
Task Execution ...

     jd_index                job_title   location_cleaned    ce_score
388       388        Software Engineer       Columbia, MD  100.000000
961       961  Automated Test Engineer  Oklahoma City, OK   89.230003
960       960        Soft

In [87]:
output_rows = []

for i in range(len(resume_df)):
    row = resume_df.iloc[i]
    top_matches = match_resume(row, jd_df, k_coarse=50, k_final=3)

    top_matches["similarity"] = top_matches["ce_score"] / 100.0

    topn = top_matches.to_dict('records')
    while len(topn) < 3:
        topn.append({
            "jd_index": np.nan,
            "job_title": "",
            "location_cleaned": "",
            "similarity": np.nan
        })

    output_rows.append({
        "resume_index": row["resume_index"],
        "resume_career_objective": row.get("career_objective", ""),
        "resume_skills": row.get("skills", ""),
        "resume_degree_names": row.get("degree_names", ""),
        "resume_major_field_of_studies": row.get("major_field_of_studies", ""),
        "resume_responsibilities": row.get("responsibilities", ""),

        "top1_jd_index": topn[0]["jd_index"],
        "top1_similarity": topn[0]["similarity"],
        "top1_job_title": topn[0]["job_title"],
        "top1_location_cleaned": topn[0]["location_cleaned"],

        "top2_jd_index": topn[1]["jd_index"],
        "top2_similarity": topn[1]["similarity"],
        "top2_job_title": topn[1]["job_title"],
        "top2_location_cleaned": topn[1]["location_cleaned"],

        "top3_jd_index": topn[2]["jd_index"],
        "top3_similarity": topn[2]["similarity"],
        "top3_job_title": topn[2]["job_title"],
        "top3_location_cleaned": topn[2]["location_cleaned"],
    })

cross_df = pd.DataFrame(output_rows)
cross_df.to_csv("crossencoder_outputs.csv", index=False)

cross_df.head()


,resume_index,resume_career_objective,resume_skills,resume_degree_names,resume_major_field_of_studies,resume_responsibilities,top1_jd_index,top1_similarity,top1_job_title,top1_location_cleaned,top2_jd_index,top2_similarity,top2_job_title,top2_location_cleaned,top3_jd_index,top3_similarity,top3_job_title,top3_location_cleaned
0,0,Experienced product development Engineer and m...,"['Microsoft Office', 'Microsoft Project', 'Pro...",['Bachelor of Science'],['Mechanical Engineering'],Management Trainee\nMechanical Systems\nMainte...,213,1.0,Product Manager,"Durham, NC",887,0.9277,Product Manager,"Boston, MA",878,0.8019,Product Analyst,"Boston, MA"
1,1,"I am a software engineer, and I want to work o...","['C++', 'Python', 'Firebase', 'Flutter', 'Tens...",['B.Tech'],['Computers'],Recruitment Coordination\nAppointment Manageme...,388,1.0,Software Engineer,"Columbia, MD",961,0.8923,Automated Test Engineer,"Oklahoma City, OK",960,0.8200,Software Engineer,"Tulsa, OK"
2,2,I desire to work for a company that provides c...,"['Word', 'SAP Time Approval', 'Excel', 'Travel...",['Associate of Arts'],['Administrative Assistance'],Machine Learning Design\nData Analysis\nModel ...,961,1.0,Automated Test Engineer,"Oklahoma City, OK",213,0.8601,Product Manager,"Durham, NC",1199,0.7615,Recruiter,"Camden, NJ"
3,3,As a Data Analyst I always look into more inno...,"['Machine Learning', 'Artificial Intelligence'...","['B.Tech', 'M.Tech']","[None, None]",Mikrotik Router Configuration\nOLT Device Setu...,944,1.0,Principal System Engineer (Big Data),"Plano, TX",213,0.9780,Product Manager,"Durham, NC",388,0.7704,Software Engineer,"Columbia, MD"
4,4,Financial and Accounting professional with exp...,['Power User of Microsoft Excel Epicor NetSuit...,['Bachelor of Business Administration'],['Accounting'],Design Creation\nCAD Drawings\nDesign Optimiza...,168,1.0,Manager,"Waltham, MA",1184,0.8251,Financial Analyst: % MBA tuition reimbursement,"Boston, MA",828,0.7107,Financial Analyst,"Louis, MO"
